# Study 804 — Realized-Kurtosis Premium — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4125, 'spread_bps': 1.78, 't_nw': 1.79, 't_1s': 1.76, 'hi_bps': 8.76, 'lo_bps': 6.97, 'welch_t': 0.69, 'gross_sharpe': 0.43, 'placebo_obs': 1.78, 'placebo_mean': -0.004, 'placebo_sd': 0.825, 'placebo_p': 0.016, 'placebo_sigma_right': 2.16, 'placebo_draws': 1000, 'era_early_bps': 0.13, 'era_early_t': 0.11, 'era_early_n': 1991, 'era_late_bps': 3.33, 'era_late_t': 2.12, 'era_late_n': 2134, 'timer_1_gross': 1.78, 'timer_1_cost': 2.14, 'timer_1_net': -0.35, 'timer_1_t': -0.35, 'timer_5_gross': 1.78, 'timer_5_cost': 10.14, 'timer_5_net': -8.35, 'timer_5_t': -8.23, 'null_mean_t': 0.35, 'null_sd_t': 0.9, 'null_fire': 0, 'planted_t': 12.81, 'planted_welch': 15.36}

## The headline — long-high-kurt / short-low-kurt spread

Daily equal-weight top-30% minus bottom-30% realized-kurtosis spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-kurt {R['hi_bps']:+.2f} vs low-kurt {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +1.78 bps/day  NW(10) t = +1.79  one-sample t = +1.76
books         : high-kurt +8.76 vs low-kurt +6.97 bps (Welch t = +0.69)
gross Sharpe  : 0.43 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

Note the tension: the placebo puts the spread ~2.2σ into the right tail (p≈0.016), but it ignores the serial correlation of the overlapping-window signal — which the Newey-West *t* (+1.79) does account for. The robust read is the sub-threshold one.

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f} "
      f"(~{R['placebo_sigma_right']:+.2f} sigma)")

observed +1.78 bps vs placebo mean -0.004 (sd 0.825) -> right-tail p = 0.01600 (~+2.16 sigma)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1991): +0.13 bps  NW t = +0.11
2018-2026 (n=2134): +3.33 bps  NW t = +2.12


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +1.78 -> net -0.35 bps/day (cost 2.14/day, t=-0.35)
5 bps one-way: gross +1.78 -> net -8.35 bps/day (cost 10.14/day, t=-8.23)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from realized_kurtosis import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=804+s, n_assets=40, n_days=1500))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.010, seed=804, n_assets=40, n_days=1500))
print(f"planted (edge=0.010): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.20 (sd 1.13), |t|>=2 in 0/8


planted (edge=0.010): NW t = +12.81, Welch t = +15.36


## Verdict

- **Signal — Weak.** The realized-kurtosis premium is, as the source paper itself reports, the **weak / ambiguous** sibling of realized skewness. On 50 liquid US mega-caps the long-high-kurt / short-low-kurt spread is **+1.78 bps/day** — the *right sign*, but NW *t* = **+1.79** (below the |t| ≥ 2 bar), the book Welch *t* is +0.69, and it is a flat zero in 2010–2017 (*t* = +0.11), surfacing only marginally in 2018–2026 (*t* = +2.12). The 20-seed synthetic control fires on 0/20 nulls and recovers a planted relation cleanly (*t* = +12.81), so this is a faithful weak measurement. Survivorship biases the magnitude.
- **Tradability — Mirage.** The +1.78 bps/day gross edge is already smaller than the 2.14 bps/day round-trip friction at 1 bp one-way (net **-0.35 bps/day**, *t* = -0.35); at 5 bps **-8.35 bps/day**.